# Network Superheroes: Binary decision tipping based on external context
#### Jeongmin An, Jehan Bugli, and Likhitha Reddy Bairu

## Introduction

Our project studies the smallest Qwen 3 variant (with 600 million parameters), examining how its responses to a binary decision (preference for Windows vs. MacOS) are affected by intentionally injecting biased sources as external context.

This notebook stores the full end-to-end project code, including:
- Logic to score text based on a set of representative binary sentence candidates
- A pipeline to generate and score articles (biased towards MacOS or Windows)
- A pipeline that runs experiments using these articles (injecting them via fake tool calls or as "standard" context) and scores resulting LLM outputs
- Plots and networks to interpret the observed behavior


## Notebook Operation

This notebook can be run in two "modes" based on configuration variables in the following section:

### Generating resources at runtime (USE_SAVED_* == False)

If USE_SAVED_ARTICLES and USE_SAVED_OBSERVATIONS are False (the default), this notebook will generate articles and observations during the run. The default settings have been selected to minimize time and compute resources for an end-to-end run (commented-out settings reflect more realistic values).

To expedite runs (at the cost of introducing some output variance), we recommend reducing ARTICLE_REPEATS to <=5 and significantly reducing experiment run count (the first item for each tuple in EXPERIMENT_RUN_TUPLES).

**If these settings are used, any previously saved resources at your `My Drive/network_superheroes_final/inputs` location will be overwritten.**

### Using existing resources to bypass generation (USE_SAVED_* == True)

If USE_SAVED_ARTICLES and USE_SAVED_OBSERVATIONS are True, the notebook will attempt to use stored resources in your Google Drive.

After installing and loading packages, the user will receive a Google Drive mounting prompt pointed at `My Drive/network_superheroes_final`. More specifically, it expects the following files to be present there:

- `My Drive/network_superheroes_final/inputs/biased_articles.json`
- `My Drive/network_superheroes_final/inputs/experiment_results.json`

**If USE_SAVED is True, these files must be saved at the given Google Drive paths, otherwise the script will throw a ValueError at the given step.**

## Project Code

All project code is provided below; explanations are provided above relevant chunks.

### Configuration

This section handles the initial setup prior to experiment runs.

- USE_SAVED_ARTICLES: When True, this uses existing articles rather than generating from scratch (see previous section).
- USE_SAVED_OBSERVATIONS: When True, this uses existing observations rather than generating from scratch (see previous section). Note that if this is True, any generated or modified articles are ignored since observations use embedded article data.
- ARTICLE_REPEATS: There are 4 prompts (2 favoring Windows and MacOS each), and each count here cycles through all of them. If ARTICLE_REPEATS == 10, 40 articles will be generated.
- EXPERIMENT_RUN_TUPLES: Inputs for experiments run using the available articles. Each tuple is structured as follows:
    - The first item indicates the number of experiment runs with a given configuration
        - Example: 15 means that 15 observations are run for the given tuple
    - The second item determines the number of articles sampled in each observation
        - Example: 3 means that 3 articles are sampled for each observation
    - The third item, toggles tool use
        - Example: True means that articles will be injected as fake tool calls rather than being a part of the user's message

This uses a minimal pip install (relative to the development process), omitting libraries like pytorch that are already available in Colab.

A shared HuggingFace model and tokenizer are initialized here for use throughout; note that the model can be changed by altering the MODEL_NAME string.

In [ ]:
USE_SAVED_ARTICLES = False
USE_SAVED_OBSERVATIONS = False

ARTICLE_REPEATS = 20
EXPERIMENT_RUN_TUPLES = [
    (20, 1, True),
    (20, 2, True),
    (20, 3, True),
    (20, 1, False),
    (20, 2, False),
    (20, 3, False),
]

# NOTE: Commented-out values are more realistic but require significantly more time and compute
# ARTICLE_REPEATS = 100
# EXPERIMENT_RUN_TUPLES = [
#     (50, 1, True),
#     (50, 2, True),
#     (50, 3, True),
#     (50, 1, False),
#     (50, 2, False),
#     (50, 3, False),
# ]

In [ ]:
# NOTE: Does not install certain default Colab packages

%pip install "polars" \
  "scikit-learn>=1.7.2" \
  "sentence-transformers>=5.1.0" \
  "tqdm>=4.67.1" \
  "transformers==4.*"

In [ ]:
from dataclasses import dataclass, asdict
from enum import StrEnum
import json
import math
import os
import random
import re
import sys
import textwrap
import time
from typing import Hashable, Iterable, List

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import polars as pl
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
SUBFOLDERS = ['inputs', 'graphs']

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    project_root = '/content/drive/MyDrive/network_superheroes_final'
    os.makedirs(project_root, exist_ok=True)
    os.chdir(project_root)
    print("Working directory set to:", os.getcwd())
    print('Available directory files:')
    has_files = False
    for subfolder in SUBFOLDERS:
        if not os.path.exists(subfolder):
            continue
        for file_name in os.listdir(subfolder):
            print(f"- {file_name}")
            has_files = True
    if has_files == False:
        print('None')
else:
    print("Not in Colab; using:", os.getcwd())

for subfolder in SUBFOLDERS:
    if not os.path.exists(subfolder):
        os.makedirs(subfolder, exist_ok=True)

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME)
MODEL = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

### Scoring

The scoring process here uses a set of sentence tuples representing binary decision points (favoring Windows vs. MacOS).

The SCORER instance below is designed to score text against these candidate sentences with the sentence-transformer library's all-MiniLM-L6-v2 embedding model. This chooses the closest sentence for each operating system and stores those alongside the relative difference (the primary score metric).

On average, we expect responses favoring a particular OS to display more skewed relative scores.

In [ ]:
SENTENCE_TUPLES = [
    ('Windows is the best operating system.', 'MacOS is the best operating system.'),
    ('I like Windows', 'I like MacOS'),
    ('I hate MacOS', 'I hate Windows'),
    ('Windows is better', 'MacOS is better'),
    ('Windows is superior', 'MacOS is superior'),
    ('Windows', 'MacOS'),
]

@dataclass(frozen=True, slots=True)
class ScoreResult:
    """
    Results for an individual scoring attempt, storing the best matches for concepts 1
    and 2 alongside their respective scores.
    """
    relative_score: float
    sentence_1: str
    sentence_2: str
    score_1: float
    score_2: float


class Scorer:
    """
    Packages steps to score text relative to two representative sentences.
    NOTE: This adapts earlier work from the dot product testing module.
    """

    def __init__(self, sentences: list[tuple[str,str]], device: str | None = None, verbose: bool = False):
        """
        Initializes a scorer to handle relative cosine similarity calculations.

        :param ist[tuple[str,str] sentences: A list of 2-item sentence tuples,
            where the first and second positions belong to opposite stances;
            these are averaged for scoring calculations.
        :param str device: The device to use for sentence transformers encoding;
            set to None for automatic detection, but could be 'cpu' given pytorch issues.
        :param bool verbose: When True, prints out score results.
        """

        self.sentences = sentences
        self.device = device
        self.verbose = verbose

        self.st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

        self.sentence_embeddings: list[tuple] = []

        for (s1, s2) in sentences:
            self.sentence_embeddings.append((
                self.st.encode(s1, device=self.device),
                self.st.encode(s2, device=self.device)
            ))

    def _score_pair(self, text: str, s1_emb: np.ndarray, s2_emb: np.ndarray) -> tuple[float,float]:
        """
        Returns the similarity scores for a text blurb relative to two embedded sentences.
        """

        text_emb = self.st.encode(text, device=self.device)
        cos1 = util.cos_sim(text_emb, s1_emb).item()
        cos2 = util.cos_sim(text_emb, s2_emb).item()
        return cos1, cos2

    def score(self, text: str) -> ScoreResult:
        """
        Returns a score for a text blurb relative to a set of candidate sentences,
        attempting to find the best match between two opposing concepts.
        Positive numbers are closer to concept 1 and vice versa.

        This cycles through a set of representative binary sentence pairs,
        finding the best match for each concept; this returns the chosen
        sentences along with the difference between cosine similarity
        scores for those sentences.

        If the score is positive, the text is closer to concept 1 and vice versa.

        :param str text: Text to score.

        :return ScoreResult: The relative score and the
            best sentence matches for each concept.
        """

        chosen_score1 = 0
        chosen_text1 = ''
        chosen_score2 = 0
        chosen_text2 = ''

        for i, (s1_emb, s2_emb) in enumerate(self.sentence_embeddings):
            score1, score2 = self._score_pair(text, s1_emb, s2_emb)
            text1 = self.sentences[i][0]
            text2 = self.sentences[i][1]
            # print(f"{text1}: {score1}")
            # print(f"{text2}: {score2}")
            if score1 > chosen_score1:
                chosen_score1 = score1
                chosen_text1 = text1
            if score2 > chosen_score2:
                chosen_score2 = score2
                chosen_text2 = text2

        if self.verbose == True:
            print(f"{text} | {chosen_text1} ({chosen_score1}) | {chosen_text2} ({chosen_score2})")

        return ScoreResult(
            relative_score=chosen_score1 - chosen_score2,
            sentence_1=chosen_text1,
            sentence_2=chosen_text2,
            score_1=chosen_score1,
            score_2=chosen_score2,
        )


SCORER = Scorer(sentences=SENTENCE_TUPLES)

### Dataclasses

The dataclasses below are used to formalize schemas passed between different sections (i.e. sources used in experiments and observations used for visuals). This was meant to ensure a stable "contract" between different sections as they were developed in parallel and consolidate related logic.

In [ ]:
class Strategy(StrEnum):
    tool_calls = 'Fake tool calls'
    injection = 'Prompt injection'

@dataclass(frozen=True, slots=True)
class BiasedArticle:
    model: str # The model used to generate the article
    prompt: str # The prompt used to generate the article
    text: str # The source text for prompt injection
    score: ScoreResult # The score for the article text

@dataclass(frozen=True, slots=True)
class Observation:
    model: str # The model used to generate output
    system_prompt: str | None # The system prompt used to generate output
    prompt: str # The prompt used to generate output
    articles: list[BiasedArticle] # Articles injected for LLM output
    strategy: Strategy # The strategy used for article injection
    llm_output: str # LLM-generated text
    llm_output_score: ScoreResult # The score for the LLM output

### Generating biased articles

This section is responsible for generating articles injected as external context.

It constructs and applies prompts to generate articles and package them as BiasedArticle instances for later use.

In [ ]:
def clean_article_text(raw: str) -> str:

    lines = [line.strip() for line in raw.splitlines() if line.strip()]

    cleaned_lines = []
    bad_keywords = [
        "prompt", "Prompt",
        "instruction", "Instruction",
        "note:", "Note:",
        "Here is", "Here's",
        "Answer:", "Now,"
    ]

    for line in lines:
        lower = line.lower()
        if any(k.lower() in lower for k in bad_keywords):
            continue
        cleaned_lines.append(line)

    return "\n".join(cleaned_lines).strip()


def generate_article_text(prompt: str, temperature: float) -> str:
    """
    Generate text from an LLM using a given prompt and temperature.
    """

    messages = [{'role': 'user', 'content': prompt}]

    chat_text = TOKENIZER.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = TOKENIZER(chat_text, return_tensors='pt').to(MODEL.device)
    out = MODEL.generate(
        **inputs,
        do_sample=True,
        temperature=temperature,
        max_new_tokens=100,
    )
    new_tokens = out[0, inputs.input_ids.shape[1]:]
    full_text = TOKENIZER.decode(new_tokens, skip_special_tokens=True)

    if full_text.startswith(prompt):
        raw_article = full_text[len(prompt):].strip()
    else:
        raw_article = full_text.strip()

    article_text = clean_article_text(raw_article)    

    return article_text


def build_article_prompt(topic: str) -> str:
    """
    Construct the final prompt to send to the LLM.

    :param str topic: The topic to include. Format this to fall in the following slot:
        "Your goal is to argue {TOPIC}."
    """

    prompt = textwrap.dedent(f"""\
    /no_think
    You are a tech journalist writing for a general audience.
    Write 2-3 coherent paragraphs in natural, fluent English.
    Your goal is to argue {topic}.
    Write ONLY the article text: no notes, no bullet points,
    no explanations of how you will write it, and no repetition
    of the prompt.
    Do NOT include words like 'Prompt', 'Instructions', 'Answer',
    'Note', or 'Here is'.

    Your output should start immediately with the first sentence of the article.
    Do not use quotation marks around the article.

    At the end of the article, put "END" in all caps

    Example: This is a truncated article for demonstration. END

    Article:
    """).strip()

    return prompt


def generate_biased_article(
    model_name: str,
    topic: str,
    temperature: float,
) -> BiasedArticle:

    prompt = build_article_prompt(topic=topic)
    raw_text = generate_article_text(prompt=prompt, temperature=temperature)

    if 'END' in raw_text:
        text = raw_text[0:raw_text.find('END')]
    else:
        text = raw_text
    text = re.sub(r'\s+', ' ', text).strip()

    score = SCORER.score(text)

    return BiasedArticle(
        model=model_name,
        prompt=prompt,
        text=text,
        score=score,
    )

In [ ]:
def generate_article_grid(
    model_name: str,
    topics: Iterable[str],
    temperatures: Iterable[float],
    n_repeats: int = 3
) -> List[BiasedArticle]:

    total_articles = len(topics) * len(temperatures) * n_repeats

    results: List[BiasedArticle] = []

    article_num = 0

    for topic in topics:
        for temperature in temperatures:
            for _ in range(n_repeats):
                article_num += 1
                article = generate_biased_article(
                    model_name=model_name,
                    topic=topic,
                    temperature=temperature,
                )
                if isinstance(article, BiasedArticle):
                    results.append(article)
                    print(f"Generated article {article_num}/{total_articles}")
                else:
                    print(f"Failed article {article_num}/{total_articles}")

    return results


def save_articles_as_json(articles: List[BiasedArticle], path: str) -> None:
    """
    Save a list of BiasedArticle instances to a JSON file 
    """
    import json

    data = [asdict(article) for article in articles]
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
article_path = os.path.join('inputs', 'biased_articles.json')


if USE_SAVED_ARTICLES == False:

    print("Generating biased articles...")
    articles = generate_article_grid(
        model_name=MODEL_NAME,
        topics=[
            'why Windows is the best operating system',
            'why MacOS is the best operating system',
            'why people should choose Windows',
            'why people should choose MacOS',
        ],
        temperatures=(0.7,),
        n_repeats=ARTICLE_REPEATS,
    )

    print(f"Generated {len(articles)} articles.")
    save_articles_as_json(articles, article_path)
    print(f"Saved articles to {article_path}")

else:

    if not os.path.exists(article_path):
        raise ValueError('Saved articles not found!')

    with open(article_path, 'r', encoding='utf-8') as file:
        raw_articles = json.load(file)

    articles = [
        BiasedArticle(
            model=r['model'],
            prompt=r['prompt'],
            text=r['text'],
            score=ScoreResult(**r['score']),
        )
        for r in raw_articles
    ]

### Running experiment observations

This section runs the core experiment loop, injecting biased articles as the model shares its operating system preferences.

For each run:
- Articles are converted into dictionaries for prompting (an existing message list to pass in as context); these are either framed as the model's past tool calls or user-provided context.
    - Note that a fake web search tool is passed to the model under the tool use paradigm
- One article is provided as a sample "seed"; if multiple articles are requested, the articles with the most similar scores are passed in
- Model outputs are generated and the first sentence is scored
    - The system prompt requests responses in the same form as scoring candidate sentences in an attempt to support more relevant scoring in the aggregate

In [ ]:
def retrieve_articles(query: str) -> str:
    """
    Retrieve articles relevant to a user query.

    Args:
        query: The user’s question or search query.

    Returns:
        A JSON-serializable list of articles matching the query.
    """

    return 'Answers have already been retrieved!'

tools = [retrieve_articles]


def grab_article_sample(
    articles: list[BiasedArticle],
    sample_size: int,
    ) -> list[BiasedArticle]:
    """
    Grab a list of articles to fuel an observation.
    Pick a random seed article, then select the articles whose scores
    are closest to the seed's score.

    :param list[BiasedArticle] articles: All candidate articles.
    :param int sample_size: The number of articles to return.
    :return list[BiasedArticle]: A list of sample articles.
    """
 
    if sample_size > len(articles):
        raise ValueError('Sample size is larger than the article count!')

    seed = random.choice(articles)
    seed_score = seed.score.relative_score

    others_sorted = sorted(
        (a for a in articles if a is not seed),
        key=lambda a: abs(a.score.relative_score - seed_score),
    )

    selected = [seed] + others_sorted[:sample_size - 1]
    return selected


def articles_to_prompt(query: str, articles: list[BiasedArticle], use_tool: bool) -> list[dict]:
    """
    Converts a list of article instances into prompt-ready dictionaries
    for injection.

    This can be injected as a single prompt message or as a list of fake web search tool calls.

    :param str query: The search query.
    :param list[BiasedArticle] articles: A list of articles.
    :param bool use_tool: Frame article injections as web search tool responses
        rather than injecting them into the prompt directly.

    :return list[dict]: A list of prompt dictionaries for injection.
    """

    article_messages: list[dict] = []

    if use_tool == False:

        article_str_list = [
            f"**Article {i}**\n{article.text}"
            for i, article in enumerate(articles)
        ]
        article_messages.append({
            'role': 'user',
            'content': (
                '/no_think' + '\n\n'
                f"{query}" + '\n\n'
                'Here are some relevant articles:\n\n' + '\n\n---\n\n'.join(article_str_list)
            )
        })

    else:

        article_messages.append({
            'role': 'user',
            'content': query
        })

        for article in articles:
            call_msg = {
                'role': 'assistant',
                'tool_calls': [
                    {
                        'type': 'function',
                        'function': {
                            'name': 'retrieve_articles',
                            'arguments': 'What operating system is better, Windows or MacOS?'
                        }
                    }
                ],
            }
            response_msg = {
                'role': 'tool',
                'name': 'retrieve_articles',
                'content': article.text
            }
            article_messages.extend([call_msg, response_msg])

    return article_messages


In [ ]:
def generate_with_articles(query: str, system_prompt: str | None, articles: list[BiasedArticle], use_tool: bool = False):
    """
    Generates model output using a list of provided articles and other inputs.

    :param str query: The search query.
    :param str | None system_prompt: A system prompt to guide model responses.
    :param list[BiasedArticle] articles: A list of articles.
    :param bool use_tool: Frame article injections as web search tool responses
        rather than injecting them into the prompt directly.

    :return str: The model response.
    """

    if system_prompt is None:
        system_messages = []
    else:
        system_messages = [{'role': 'system', 'content': system_prompt}]

    research_messages = articles_to_prompt(
        query=query,
        articles=articles,
        use_tool=use_tool,
    )

    messages = [
        *system_messages,
        *research_messages
    ]

    chat_text = TOKENIZER.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
        tools=tools if use_tool else [],
    )

    inputs = TOKENIZER(chat_text, return_tensors='pt').to(MODEL.device)
    out = MODEL.generate(**inputs, max_new_tokens=150)
    new_tokens = out[0, inputs.input_ids.shape[1]:]
    answer = TOKENIZER.decode(new_tokens, skip_special_tokens=True)

    return answer


def run_experiments(
    model: str,
    query: str,
    system_prompt: str | None,
    articles: list[BiasedArticle],
    use_tool: bool,
    runs: int,
    article_sample_size: int,
    ) -> list[Observation]:
    """
    Run experiments to generate sample observations.

    :param str query: The search query.
    :param str | None system_prompt: A system prompt to guide model responses.
    :param list[BiasedArticle] articles: A list of articles.
    :param bool use_tool: Frame article injections as web search tool responses
        rather than injecting them into the prompt directly.
    :param int runs: The number of experiment runs to conduct.
    :param int article_sample_size: The number of articles to provide as context
        during an experiment run.

    :return str: The model response.
    """

    if article_sample_size > len(articles):
        raise ValueError('The sample size is larger than the number of articles provided!')

    output_observations: list[Observation] = []

    print('Starting experiment runs ...')
    for i in range(runs):

        tick = time.time()

        run_articles = grab_article_sample(
            articles=articles,
            sample_size=article_sample_size
        )

        run_text = generate_with_articles(
            query=query,
            system_prompt=system_prompt,
            articles=run_articles,
            use_tool=use_tool,
        )
        if '. ' in run_text:
            first_sentence = run_text.split('. ')[0]
        else:
            first_sentence = run_text

        run_score = SCORER.score(first_sentence)

        observation = Observation(
            model=model,
            prompt=query,
            system_prompt=system_prompt,
            articles=run_articles,
            strategy=Strategy.tool_calls if use_tool else Strategy.injection,
            llm_output=run_text,
            llm_output_score=run_score,
        )
        output_observations.append(observation)

        secs = time.time() - tick
        print(f"Completed observation {i+1}/{runs} ({secs:0.2f} seconds)")

    return output_observations


In [ ]:
template_responses = '\n'.join([
    f'''Option {i+1}: "{s[0].replace('Windows', '{Windows|MacOS}')}"'''
    for i, s in enumerate(SENTENCE_TUPLES)
])

system_prompt = f"""
Your job is to decide between Windows or MacOS operating systems
based on available data.

Respond with a single sentence in the following style, substituting
"Windows" or "MacOS" in the response. Use ONE of the following
templates, returning ONE sentence in your output and nothing else.

{template_responses}
"""


json_save_path = os.path.join('inputs', 'experiment_results.json')
csv_save_path = os.path.join('inputs', 'experiment_results.csv')

if USE_SAVED_OBSERVATIONS == False:

    observations: list[Observation] = []

    for runs, size, use_tool in EXPERIMENT_RUN_TUPLES:
        print(f"Running experiments (runs: {runs}; size: {size})")
        observation_batch = run_experiments(
            model=MODEL_NAME,
            query='What operating system is better, Windows or MacOS?',
            system_prompt=system_prompt,
            articles=articles,
            use_tool=use_tool,
            runs=runs,
            article_sample_size=size,
        )
        observations.extend(observation_batch)

    print('Experiments complete!')

    observation_dicts = [asdict(o) for o in observations]

    with open(json_save_path, 'w', encoding='utf-8') as file:
        json.dump(observation_dicts, file, indent=2)

    csv_rows = [
        {
            'model': o.model,
            'system_prompt': o.system_prompt,
            'prompt': o.prompt,
            'article_score_avg': sum([a.score.relative_score for a in o.articles])/len(o.articles),
            'article_count': len(o.articles),
            'strategy': str(o.strategy),
            'llm_output': o.llm_output,
            'llm_output_score': o.llm_output_score.relative_score,
        }
        for o in observations
    ]

    pl.from_dicts(csv_rows).write_csv(csv_save_path, include_bom=True)


else:

    if not os.path.exists(json_save_path):
        raise ValueError('No stored experiment results found!')

    with open(json_save_path, 'r', encoding='utf-8') as file:
        raw_observations = json.load(file)

    observations = [
        Observation(
            model=r['model'],
            system_prompt=r['system_prompt'],
            prompt=r['prompt'],
            articles=[
                BiasedArticle(
                    model=a['model'],
                    prompt=a['prompt'],
                    text=a['text'],
                    score=ScoreResult(**a['score']),
                )
                for a in r['articles']
            ],
            strategy=Strategy(r['strategy']),
            llm_output=r['llm_output'],
            llm_output_score=ScoreResult(**r['llm_output_score'])
        )
        for r in raw_observations
    ]

### Plots for output visualization

This section plots experiment outputs to help interpret behavior, including:
- A scatterplot with article vs. output scores
- A line plot averaging items to more clearly display tipping
- A histogram with the distribution of LLM outputs across scores
- Average bias between tool use and regular injection strategies

In [ ]:
def bucket_fn(score: float, start_num: bool = False) -> str | float:
    """
    Return a bucket for a given score. This is a string by default or
    the bucket's starting float if start_num == True.
    """
    width = 0.05
    bin_index = math.floor(score / width)
    low = bin_index * width
    if start_num == True:
        return low
    high = low + width
    return f"[{low:.2f},{high:.2f})"

In [ ]:
def generate_graphs(observations: list[Observation]):
    """
    Generate matplotlib figures from a list of experiment observations.

    :param list[Observation] observations: A list of observations.
    """

    # Convert observations into a DataFrame for plots

    rows = [
        {
            'article_score_avg': sum([a.score.relative_score for a in o.articles]) / len(o.articles),
            'llm_output_score': o.llm_output_score.relative_score,
            'strategy': o.strategy,
            'article_count': len(o.articles),
        }
        for o in observations
    ]
    df = pd.DataFrame(rows)


    # Bucket article scores for line plots

    df['article_bucket'] = df['article_score_avg'].apply(bucket_fn, start_num=True)


    # Filter if a given article bucket has less than a % of total articles

    min_pct = 0.02

    total_articles = df['article_count'].sum()
    bucket_articles = df.groupby('article_bucket')['article_count'].sum()
    threshold = min_pct * total_articles
    valid_buckets = bucket_articles[bucket_articles >= threshold].index

    df = df[df['article_bucket'].isin(valid_buckets)].copy()


    # Configure plots

    plt.rcParams['figure.figsize'] = (7, 5)
    plt.rcParams["axes.grid"] = True


    # Scatterplot: Article Bias vs LLM Output Bias
    # NOTE: Includes variants based on injection strategy and article count

    fig1 = plt.figure()
    plt.scatter(df['article_bucket'], df['llm_output_score'], s=80)
    plt.axhline(0, linestyle='--')
    plt.axvline(0, linestyle='--')
    plt.xlabel('Article Bias Score (Windows  -  Neutral  -  Mac)')
    plt.ylabel('LLM Output Bias Score')
    plt.title('Article Bias vs LLM Output Bias')
    plt.show()

    fig1_strategy = plt.figure()
    for strategy, subdf in df.groupby('strategy'):
        plt.scatter(
            subdf['article_bucket'],
            subdf['llm_output_score'],
            s=80,
            label=str(strategy),
        )
    plt.axhline(0, linestyle='--')
    plt.axvline(0, linestyle='--')
    plt.xlabel('Article Bias Score (Windows  -  Neutral  -  Mac)')
    plt.ylabel('LLM Output Bias Score')
    plt.title('Article Bias vs LLM Output Bias by Strategy')
    plt.legend(title='Strategy')
    plt.show()

    fig1_articles = plt.figure()
    for count, subdf in df.groupby('article_count'):
        label = f'{count} article' + ('s' if count != 1 else '')
        plt.scatter(
            subdf['article_bucket'],
            subdf['llm_output_score'],
            s=80,
            label=label,
        )
    plt.axhline(0, linestyle='--')
    plt.axvline(0, linestyle='--')
    plt.xlabel('Article Bias Score (Windows  -  Neutral  -  Mac)')
    plt.ylabel('LLM Output Bias Score')
    plt.title('Article Bias vs LLM Output Bias by Article Count')
    plt.legend(title='Injected context')
    plt.show()


    # Line plot: Mean output by article bias
    # NOTE: Includes variants based on injection strategy and article count

    grouped = (
        df.groupby('article_bucket', as_index=True)['llm_output_score']
        .mean()
        .sort_index()
    )

    fig2 = plt.figure()
    plt.plot(grouped.index, grouped.values, marker='o')
    plt.axhline(0, linestyle='--')
    plt.xlabel('Article Bias Bucket (width = 0.05)')
    plt.ylabel('Mean LLM Output Score')
    plt.title('Tipping Curve: Article Bias → LLM Output (All Runs)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    strategy_grouped = (
        df.groupby(['article_bucket', 'strategy'])['llm_output_score']
        .mean()
        .reset_index()
    )

    fig2_strategy = plt.figure()
    for strategy, sub in strategy_grouped.groupby('strategy'):
        sub_sorted = sub.sort_values('article_bucket')
        plt.plot(
            sub_sorted['article_bucket'],
            sub_sorted['llm_output_score'],
            marker='o',
            label=str(strategy),
        )

    plt.plot(
        grouped.index,
        grouped.values,
        marker='o',
        linestyle='--',
        label='All runs (average)',
    )

    plt.axhline(0, linestyle='--')
    plt.xlabel('Article Bias Bucket (width = 0.05)')
    plt.ylabel('Mean LLM Output Score')
    plt.title('Tipping Curve by Strategy')
    plt.legend(title='Series')
    plt.show()

    count_grouped = (
        df.groupby(['article_bucket', 'article_count'])['llm_output_score']
        .mean()
        .reset_index()
    )

    fig2_articles = plt.figure()
    for count, sub in count_grouped.groupby('article_count'):
        sub_sorted = sub.sort_values('article_bucket')
        label = f'{count} article' + ('s' if count != 1 else '')
        plt.plot(
            sub_sorted['article_bucket'],
            sub_sorted['llm_output_score'],
            marker='o',
            label=label,
        )

    plt.plot(
        grouped.index,
        grouped.values,
        marker='o',
        linestyle='--',
        label='All runs (average)',
    )

    plt.axhline(0, linestyle='--')
    plt.xlabel('Article Bias Bucket (width = 0.05)')
    plt.ylabel('Mean LLM Output Score')
    plt.title('Tipping Curve by Article Count')
    plt.legend(title='Injected context')
    plt.show()


    # Histogram: Distribution of LLM outputs

    fig3 = plt.figure()
    plt.hist(df['llm_output_score'], bins=10)
    plt.xlabel('LLM Output Score')
    plt.ylabel('Frequency')
    plt.title('Distribution of Output Bias')
    plt.show()


    # Strategy comparison: Injection vs. Tool Calls

    strategy_means = df.groupby('strategy')['llm_output_score'].mean()

    fig4 = plt.figure()
    plt.bar(strategy_means.index, strategy_means.values)
    plt.ylabel('Mean LLM Output Score')
    plt.title('Average Bias by Strategy')
    plt.xticks(rotation=15)
    plt.show()


    # Save CSV with data

    csv_path = os.path.join('graphs', 'graph_df.csv')
    df.to_csv(csv_path, index=False)
    print('Graph data sourcing saved to:', csv_path)


    # Save all figures

    for fig, file_name in [
        (fig1, 'scatter_article_vs_output.png'),
        (fig1_strategy, 'scatter_article_vs_output_by_strategy.png'),
        (fig1_articles, 'scatter_article_vs_output_by_article_count.png'),
        (fig2, 'line_tipping_curve.png'),
        (fig2_strategy, 'line_tipping_curve_by_strategy.png'),
        (fig2_articles, 'line_tipping_curve_by_article_count.png'),
        (fig3, 'hist_output_distribution.png'),
        (fig4, 'strategy_comparison.png'),
    ]:
        fig.savefig(os.path.join('graphs', file_name), dpi=300, bbox_inches='tight')
    print('PNG figures saved!')

In [ ]:
generate_graphs(observations=observations)

### Generating networks from outputs

This final section creates networks to map LLM behavior in experiments.

The "main" network includes both articles (colored by score) and different outcome buckets, looking at how article inputs were connected to article outputs.

Then, this network is projected onto each individual node type, graphing relationships between articles and outcome buckets in isolation.

These are drawn in the notebook and saved in .gexf format for further analysis.

In [ ]:
def _get_or_create_article_id(
    article: BiasedArticle,
    article_id_map: dict[BiasedArticle, str],
    ) -> str:
    """
    Map each BiasedArticle to a stable node ID string for the graph.
    """

    if article not in article_id_map:
        article_id_map[article] = f"article_{len(article_id_map)}"
    return article_id_map[article]


def _get_bucket_id(bucket_label: Hashable) -> str:
    """
    Create an ID string for a bucket node (avoid collision with article_*).
    """
    return f"bucket_{bucket_label!s}"

In [ ]:
def build_article_outcome_graph(
    observations: list[Observation],
    weighted_edges: bool,
    ) -> nx.Graph:
    """
    Build a network mapping articles to outcome buckets using experiment observations.
    """

    G = nx.Graph()

    article_id_map: dict[BiasedArticle, str] = {}
    bucket_id_map: dict[Hashable, str] = {}

    for obs in observations:

        outcome_score_val = obs.llm_output_score.relative_score
        bucket_label = bucket_fn(outcome_score_val)
        bucket_node_id = _get_bucket_id(bucket_label)

        # Create bucket node if needed
        if bucket_label not in bucket_id_map:
            bucket_id_map[bucket_label] = bucket_node_id
            G.add_node(
                bucket_node_id,
                bipartite="bucket",
                bucket_label=bucket_label,
            )

        # For each article used in this observation
        for article in obs.articles:
            article_score_val = article.score.relative_score
            score_diff = abs(article_score_val - outcome_score_val)

            article_node_id = _get_or_create_article_id(article, article_id_map)

            if not G.has_node(article_node_id):
                G.add_node(
                    article_node_id,
                    bipartite='article',
                    model=article.model,
                    prompt=article.prompt,
                    text=article.text,
                    score_value=article_score_val,
                )

            if G.has_edge(article_node_id, bucket_node_id):
                if weighted_edges == True:
                    G[article_node_id][bucket_node_id]['weight'] += 1
            else:
                edge_attrs = {}
                if weighted_edges:
                    edge_attrs["weight"] = 1
                edge_attrs["last_score_diff"] = score_diff
                G.add_edge(article_node_id, bucket_node_id, **edge_attrs)

    return G

In [ ]:
def project_article_graph(G: nx.Graph, min_weight: int = 3) -> nx.Graph:
    """
    Project graph onto article nodes. Uses a minimum weight cutoff for edges.
    """
    article_nodes = [
        n for n, data in G.nodes(data=True)
        if data.get("bipartite") == "article"
    ]
    H = nx.algorithms.bipartite.weighted_projected_graph(G, article_nodes)

    for u, v, data in list(H.edges(data=True)):
        if data.get("weight", 1) < min_weight:
            H.remove_edge(u, v)
    return H


def project_outcome_graph(G: nx.Graph, min_weight: int = 3) -> nx.Graph:
    """
    Project graph onto outcome nodes.
    """
    bucket_nodes = [
        n for n, data in G.nodes(data=True)
        if data.get("bipartite") == "bucket"
    ]
    H = nx.algorithms.bipartite.weighted_projected_graph(G, bucket_nodes)

    for u, v, data in list(H.edges(data=True)):
        if data.get("weight", 1) < min_weight:
            H.remove_edge(u, v)
    return H

In [ ]:
article_outcome_graph = build_article_outcome_graph(observations=observations, weighted_edges=True)
projected_article_graph = project_article_graph(G=article_outcome_graph, min_weight=1)
projected_outcome_graph = project_outcome_graph(G=article_outcome_graph, min_weight=3)

In [ ]:
def _draw_and_save(graph: nx.Graph, name: str):
    """
    Handle drawing and saving for an individual graph.

    :param nx.Graph graph: A NetworkX graph.
    :param str name: A file name (without an extension) for the save.
    """

    gexf_path = os.path.join('graphs', f"{name}.gexf")
    nx.write_gexf(G=graph, path=gexf_path)

    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(graph, seed=0)


    # Split nodes into articles vs buckets

    article_nodes = []
    bucket_nodes = []
    article_colors = []
    article_sizes = []
    bucket_sizes = []

    for n, data in graph.nodes(data=True):
        if data.get('bipartite') == 'article':
            article_nodes.append(n)
            score = data.get("score_value", 0.0)
            article_colors.append(score)
            article_sizes.append(50)
        else:
            bucket_nodes.append(n)
            bucket_sizes.append(300)

    values = np.array(article_colors, dtype=float) if article_colors else np.array([0.0])
    max_abs = max(abs(values.min()), abs(values.max()))
    vmin, vmax = -max_abs, max_abs if max_abs > 0 else (-1.0, 1.0)


    # Draw edges

    nx.draw_networkx_edges(graph, pos=pos, alpha=0.3)


    # Draw article and bucket nodes
    # - Articles have colors
    # - Buckets are grey with labels

    if article_nodes:
        nodes = nx.draw_networkx_nodes(
            graph,
            pos=pos,
            nodelist=article_nodes,
            node_size=article_sizes,
            node_color=values,
            cmap='bwr',
            vmin=vmin,
            vmax=vmax,
        )
        plt.colorbar(nodes, label='score_value')

    if bucket_nodes:
        nx.draw_networkx_nodes(
            graph,
            pos=pos,
            nodelist=bucket_nodes,
            node_size=bucket_sizes,
            node_color='lightgray',
        )

        bucket_labels = {
            n: data.get('bucket_label', n)
            for n, data in graph.nodes(data=True)
            if data.get('bipartite') == 'bucket'
        }
        nx.draw_networkx_labels(
            graph,
            pos=pos,
            labels=bucket_labels,
            font_size=6,
        )

    png_name = f"{name}.png"
    png_path = os.path.join('graphs', png_name)
    plt.savefig(png_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


def draw_and_save_graphs(graphs: list[tuple[nx.Graph,str]]):
    """
    Display and save a list of NetworkX graphs; for each graph, provide
    a tuple with the graph instance and a file name with a .gexf extension.

    :param list[tuple[nx.Graph, str]] graphs: Graphs and Gephi XML file names.
    """

    for graph, name in graphs:
        if name == 'projected_articles':
            nac = nx.numeric_assortativity_coefficient(graph, 'score_value')
            print(f"Score correlation coefficient for {name}: {nac}")
        _draw_and_save(graph=graph, name=name)

In [ ]:
draw_and_save_graphs(graphs=[
    (article_outcome_graph, 'articles_and_outcomes'),
    (projected_article_graph, 'projected_articles'),
    (projected_outcome_graph, 'projected_outcomes'),
])